In [1]:
import pandas as pd
import sys
sys.path.insert(0, "..")        # Search for utils.py module on project's root
from utils import fill_empty_rows, import_datasets_to_db, create_file_path

In [29]:
# Read the .csv file
df_gm = pd.read_csv(create_file_path("gmR.csv"))

In [30]:
# Check the df info
df_gm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4298 entries, 0 to 4297
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ModType       2886 non-null   object
 1   Hardware      2881 non-null   object
 2   GM CAL №      2834 non-null   object
 3   ID            2448 non-null   object
 4   P/N           2873 non-null   object
 5   Bosch 0261 №  1947 non-null   object
 6   VIN Example   3458 non-null   object
 7   EPC PN        3180 non-null   object
 8   Year          3540 non-null   object
 9   Make & Model  3558 non-null   object
 10  Engine        3317 non-null   object
 11  Prog Info     835 non-null    object
dtypes: object(12)
memory usage: 403.1+ KB


In [31]:
# Remove rows where all cols have null (NaN) values,
# keeping only rows with at least one valid (non-null) entry.
df_gm_not_allNull = df_gm[~df_gm.isnull().all(axis=1)]

In [32]:
# Keep only rows where more than one col has a non-null value.
# This removes rows with only a single non-null entry, ensuring each row has multiple valid data points.
df_gm_more_than_one_not_null = df_gm_not_allNull[df_gm_not_allNull.notnull().sum(axis=1) != 1]

In [33]:
# Fill null values in the 'Hardware' col by propagating the last valid value forward.
# method='ffill' -> copies the value from the previous row into rows with null values.
df_gm_more_than_one_not_null['Hardware'] = df_gm_more_than_one_not_null['Hardware'].fillna(method='ffill')

C:\Users\Weverson Barbieri\AppData\Local\Temp\ipykernel_22172\1009899977.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_gm_more_than_one_not_null['Hardware'] = df_gm_more_than_one_not_null['Hardware'].fillna(method='ffill')
C:\Users\Weverson Barbieri\AppData\Local\Temp\ipykernel_22172\1009899977.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gm_more_than_one_not_null['Hardware'] = df_gm_more_than_one_not_null['Hardware'].fillna(method='ffill')


In [34]:
# Remove all lines with the repeated cols
repeated_cols_index_list = df_gm_more_than_one_not_null[df_gm_more_than_one_not_null['ModType'] == 'ModType'].index

In [35]:
# Remove all lines with repeated cols
df_gm_rm_repeated_cols = df_gm_more_than_one_not_null.drop(index=repeated_cols_index_list)

In [36]:
# Map all cols to be deleted
col_map = ['ModType', 'ID', 'VIN Example', 'Year', 'Make & Model', 'Engine', 'Prog Info']

In [37]:
# Remove no needed cols according to the map
df_gm_rm_cols = df_gm_rm_repeated_cols.drop(columns=col_map)

In [38]:
# Show the cols
df_gm_rm_cols.columns

Index(['Hardware', 'GM CAL №', 'P/N', 'Bosch 0261 №', 'EPC PN'], dtype='object')

In [39]:
# Map cols will be renamed
rename_mapping_cols = {
    "P/N": "Service №",
    "Bosch 0261 №": "Other PN"
}

In [40]:
# Rename cols from P/N and Bosch 0261 №
df_gm_rn_cols = df_gm_rm_cols.rename(columns=rename_mapping_cols)

In [41]:
# Call the function to fill out null values
df_gm_fill_null_values = fill_empty_rows(df_gm_rn_cols)

In [42]:
# Print the cols
df_gm_fill_null_values.columns

Index(['Hardware', 'GM CAL №', 'Service №', 'Other PN', 'EPC PN'], dtype='object')

### Call the function to the explode all items of the cols

In [43]:
# Reset the index
df_gm_reset_index = df_gm_fill_null_values.reset_index(drop=True)

In [44]:
# Check the df info
df_gm_reset_index.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4014 entries, 0 to 4013
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Hardware   4014 non-null   object
 1   GM CAL №   4014 non-null   object
 2   Service №  4014 non-null   object
 3   Other PN   4014 non-null   object
 4   EPC PN     4014 non-null   object
dtypes: object(5)
memory usage: 156.9+ KB


In [45]:
# Show the column names
df_gm_reset_index.columns

Index(['Hardware', 'GM CAL №', 'Service №', 'Other PN', 'EPC PN'], dtype='object')

In [47]:
# Dict to rename the columns
rn_col_map = {
    "Hardware": "hardware",
    "GM CAL №": "pn1",
    "Service №": "pn2",
    "Other PN": "pn3",
    "EPC PN": "pn4"
}

In [48]:
# Rename the cols applying the map
df_gm_rn_cols = df_gm_reset_index.rename(columns=rn_col_map)

In [50]:
# Check the df info
df_gm_rn_cols.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4014 entries, 0 to 4013
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   hardware  4014 non-null   object
 1   pn1       4014 non-null   object
 2   pn2       4014 non-null   object
 3   pn3       4014 non-null   object
 4   pn4       4014 non-null   object
dtypes: object(5)
memory usage: 156.9+ KB


In [51]:
# Call the function to import the df cleaned to the postgreSQLdb
import_datasets_to_db(df_gm_rn_cols, 'part_numbers_gm')

14